In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\ShawRd_pump")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\ShawRd_pump


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_ShawRd_pump_before.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
sQ2106,3.57,23.96,0.043
sQ2107,4.72,16.99,0.068
sQ2108,4.15,2.86,0.040
sQ2108A,4.28,2.78,0.040
sQ2109,4.56,10.90,0.054


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_ShawRd_pump_after.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
sQ2106,0.01,11.75,0.000
sQ2107,2.47,14.13,0.009
sQ2108,0.14,1.11,0.001
sQ2108A,0.30,1.05,0.002
sQ2109,2.55,1.71,0.004


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
comparison["Pre Total Flood Vol (MG)"] = comparison["Pre Total Flood Vol (MG)"].fillna(0)
comparison["Post Total Flood Vol (MG)"] = comparison["Post Total Flood Vol (MG)"].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

# Percent reduction (simple)
comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction
Node,,,,,,,,
ShawRdPumpStation,2.87,11.02,0.453,2.18,38.10,0.544,-0.091,-20.1
sQ2107,4.72,16.99,0.068,2.47,14.13,0.009,0.059,86.8
sQ2109,4.56,10.90,0.054,2.55,1.71,0.004,0.050,92.6
sQ2109A,3.84,4.71,0.045,0.01,0.38,0.000,0.045,100.0
sQ2106,3.57,23.96,0.043,0.01,11.75,0.000,0.043,100.0
sQ2110,4.11,6.77,0.042,0.16,0.85,0.001,0.041,97.6
sQ2108,4.15,2.86,0.040,0.14,1.11,0.001,0.039,97.5
sQ2108A,4.28,2.78,0.040,0.30,1.05,0.002,0.038,95.0


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

Pre-Project Network Total Flood Volume: 0.785 MG
Post-Project Network Total Flood Volume: 0.561 MG
Network Flood Volume Reduction: 0.224 MG
Network Flood Volume Percent Reduction: 28.5%


In [8]:
# Fix: ensure this is a scalar (not numpy array)
network_percent_reduction = float(network_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)"
    ],
    "Value": [
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")# Fix: ensure this is a scalar (not numpy array)

Exported: network_flood_summary.csv
